In [ ]:
storage_account = "stpdmvvpzsphfeeck72vs"
storage_key = dbutils.secrets.get(scope="pdm-vvp", key="storage-account-key")

In [ ]:
def read_bronze(path, fmt):
    reader = spark.read.option(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
    if fmt == "csv":
        reader = reader.option("header", True).option("inferSchema", True)
    return getattr(reader, fmt)(f"abfss://bronze@{storage_account}.dfs.core.windows.net/{path}/")

def write_silver(df, name):
    df.write.format("delta").mode("overwrite").saveAsTable(f"silver_{name}")

In [ ]:
from pyspark.sql.functions import col

# Machines
df_machines_silver = (
    read_bronze("machines", "parquet")
    .dropDuplicates(["machineID"])
    .withColumn("machineID", col("machineID").cast("int"))
    .withColumn("age", col("age").cast("int"))
)
write_silver(df_machines_silver, "machines")

# Maintenance history
df_maint_silver = (
    read_bronze("maintenance", "parquet")
    .dropDuplicates()
    .withColumn("datetime", col("datetime").cast("timestamp"))
    .withColumn("machineID", col("machineID").cast("int"))
)
write_silver(df_maint_silver, "maintenance")

# Telemetry
df_telemetry_silver = (
    read_bronze("telemetry", "csv")
    .dropDuplicates()
    .withColumn("datetime", col("datetime").cast("timestamp"))
    .withColumn("machineID", col("machineID").cast("int"))
    .withColumn("volt", col("volt").cast("double"))
    .withColumn("rotate", col("rotate").cast("double"))
    .withColumn("pressure", col("pressure").cast("double"))
    .withColumn("vibration", col("vibration").cast("double"))
    .na.drop(subset=["volt", "rotate", "pressure", "vibration"])
)
write_silver(df_telemetry_silver, "telemetry")

# Errors
df_errors_silver = (
    read_bronze("errors", "csv")
    .dropDuplicates()
    .withColumn("datetime", col("datetime").cast("timestamp"))
    .withColumn("machineID", col("machineID").cast("int"))
)
write_silver(df_errors_silver, "errors")

# Failures
df_failures_silver = (
    read_bronze("failures", "csv")
    .dropDuplicates()
    .withColumn("datetime", col("datetime").cast("timestamp"))
    .withColumn("machineID", col("machineID").cast("int"))
)
write_silver(df_failures_silver, "failures")

In [ ]:
for name in ["machines", "maintenance", "telemetry", "errors", "failures"]:
    df = spark.table(f"silver_{name}")
    print(name, df.count())

In [ ]:
from pyspark.sql.functions import to_date, avg, stddev
from pyspark.sql.window import Window

silver_telemetry = spark.table("silver_telemetry")

daily_telemetry = (
    silver_telemetry
    .withColumn("date", to_date("datetime"))
    .groupBy("machineID", "date")
    .agg(
        avg("volt").alias("volt_mean"),
        avg("rotate").alias("rotate_mean"),
        avg("pressure").alias("pressure_mean"),
        avg("vibration").alias("vibration_mean"),
    )
)

window_3day = Window.partitionBy("machineID").orderBy("date").rowsBetween(-2, 0)

gold_telemetry_features = (
    daily_telemetry
    .withColumn("volt_roll3_mean", avg("volt_mean").over(window_3day))
    .withColumn("volt_roll3_std", stddev("volt_mean").over(window_3day))
    .withColumn("rotate_roll3_mean", avg("rotate_mean").over(window_3day))
    .withColumn("rotate_roll3_std", stddev("rotate_mean").over(window_3day))
    .withColumn("pressure_roll3_mean", avg("pressure_mean").over(window_3day))
    .withColumn("pressure_roll3_std", stddev("pressure_mean").over(window_3day))
    .withColumn("vibration_roll3_mean", avg("vibration_mean").over(window_3day))
    .withColumn("vibration_roll3_std", stddev("vibration_mean").over(window_3day))
)

gold_telemetry_features.write.format("delta").mode("overwrite").saveAsTable("gold_telemetry_features")

In [ ]:
df = spark.table("gold_telemetry_features")
print(df.count())
df.orderBy("machineID", "date").show(10)

In [ ]:
from pyspark.sql.functions import count

silver_errors = spark.table("silver_errors")

daily_errors = (
    silver_errors
    .withColumn("date", to_date("datetime"))
    .groupBy("machineID", "date")
    .agg(count("*").alias("error_count"))
)

In [ ]:
from pyspark.sql import functions as F

silver_maintenance = spark.table("silver_maintenance")

maint_dates = (
    silver_maintenance
    .withColumn("date", to_date("datetime"))
    .select("machineID", "date")
    .distinct()
    .withColumnRenamed("date", "maint_date")
)

grid = spark.table("gold_telemetry_features").select("machineID", "date")

grid_joined = grid.join(
    maint_dates,
    (grid.machineID == maint_dates.machineID) & (grid.date == maint_dates.maint_date),
    "left"
).select(grid["machineID"], grid["date"], maint_dates["maint_date"])

w = Window.partitionBy("machineID").orderBy("date")

maintenance_recency = (
    grid_joined
    .withColumn("last_maint_date", F.last("maint_date", ignorenulls=True).over(w))
    .withColumn("days_since_maintenance", F.datediff("date", "last_maint_date"))
    .select("machineID", "date", "days_since_maintenance")
)

In [ ]:
silver_failures = spark.table("silver_failures")

failures_next_day = (
    silver_failures
    .withColumn("failure_date", to_date("datetime"))
    .withColumn("date", F.date_sub("failure_date", 1))
    .select(
        "machineID",
        "date",
        F.lit(1).alias("failure_next_day"),
        col("failure").alias("failure_component"),
    )
)

In [ ]:
gold_features = (
    spark.table("gold_telemetry_features")
    .join(daily_errors, on=["machineID", "date"], how="left")
    .join(maintenance_recency, on=["machineID", "date"], how="left")
    .join(failures_next_day, on=["machineID", "date"], how="left")
    .fillna({"error_count": 0, "failure_next_day": 0})
)

gold_features.write.format("delta").mode("overwrite").saveAsTable("gold_features")

In [ ]:
df = spark.table("gold_features")
print(df.count())
df.filter(col("failure_next_day") == 1).show(10)

In [ ]:
from pyspark.sql.functions import current_timestamp

def run_dq_checks(table_name):
    df = spark.table(table_name)
    total_rows = df.count()

    results = [(table_name, "row_count", str(total_rows), True)]

    for field in df.schema.fields:
        col_name = field.name
        null_count = df.filter(col(col_name).isNull()).count()
        null_pct = round(null_count / total_rows * 100, 2) if total_rows > 0 else 0
        passed = null_pct < 5  # flag anything with more than 5% nulls
        results.append((table_name, f"null_pct_{col_name}", f"{null_pct}%", passed))

    return results

In [ ]:
tables_to_check = [
    "silver_machines", "silver_maintenance", "silver_telemetry",
    "silver_errors", "silver_failures", "gold_features",
]

all_results = []
for t in tables_to_check:
    all_results.extend(run_dq_checks(t))

dq_df = (
    spark.createDataFrame(all_results, ["table_name", "check_name", "result", "passed"])
    .withColumn("run_timestamp", current_timestamp())
)

dq_df.write.format("delta").mode("append").saveAsTable("dq_check_log")
dq_df.filter(~col("passed")).show(50, truncate=False)

In [ ]:
data_dictionary = spark.createDataFrame([
    ("gold_features", "machineID", "int", "Unique machine identifier"),
    ("gold_features", "date", "date", "Calendar day of the observation"),
    ("gold_features", "volt_roll3_mean", "double", "3-day rolling mean voltage — precursor signal for Comp1 failures"),
    ("gold_features", "rotate_roll3_mean", "double", "3-day rolling mean rotation speed — precursor signal for Comp2 failures"),
    ("gold_features", "pressure_roll3_mean", "double", "3-day rolling mean pressure — precursor signal for Comp3 failures"),
    ("gold_features", "vibration_roll3_mean", "double", "3-day rolling mean vibration — precursor signal for Comp4 failures"),
    ("gold_features", "error_count", "int", "Count of error codes logged for this machine on this day"),
    ("gold_features", "days_since_maintenance", "int", "Days elapsed since the most recent maintenance event, any component"),
    ("gold_features", "failure_next_day", "int", "Label: 1 if any component fails on the following day, else 0"),
    ("gold_features", "failure_component", "string", "Which component failed, if failure_next_day = 1"),
], ["table_name", "column_name", "data_type", "description"])

data_dictionary.write.format("delta").mode("overwrite").saveAsTable("data_dictionary")
data_dictionary.show(truncate=False)